# Botanist ML — Final training

Train a custom four-layer PyTorch CNN for 38-class leaf image classification. This notebook contains the final full-dataset training configuration. The development split, validation runs, and competition inference are documented separately where evidence is available.


## Setup

Use Python with PyTorch, torchvision, pandas, and Pillow. Extract the course-provided images into `TrainFiles/` and place `Botanist_Training_Set.csv` beside that directory. Set `DATA_DIR` to their parent directory. The loader expects the first two columns to contain the filename stem and label. In Colab, a GPU runtime is useful for training.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
import pandas as pd
from PIL import Image
import os
from pathlib import Path

DATA_DIR = Path.cwd()
CHECKPOINT_PATH = Path.cwd() / "botanist_final_model.pth"


In [ ]:
# ==========================================
# 1. Final Dataset Loader (Uses 100% of data)
# ==========================================
class BotanistDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.img_labels = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        # Construct image path
        img_name = os.path.join(self.img_dir, str(self.img_labels.iloc[idx, 0]) + '.jpg')
        # Load image and ensure RGB
        image = Image.open(img_name).convert("RGB")
        # Label: map 1-38 to 0-37
        label = int(self.img_labels.iloc[idx, 1]) - 1

        if self.transform:
            image = self.transform(image)

        return image, label

# Data Augmentation Strategy
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    # Data Augmentation techniques
    transforms.RandomHorizontalFlip(p=0.5), # 50% probability flip
    transforms.RandomVerticalFlip(p=0.5),   # 50% probability flip
    transforms.RandomRotation(degrees=15),  # Rotate +/- 15 degrees
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05), # Simulate lighting
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Initialize Dataset
full_dataset = BotanistDataset(csv_file=DATA_DIR / 'Botanist_Training_Set.csv', img_dir=DATA_DIR / 'TrainFiles', transform=train_transform)

# DataLoader (using full dataset)
train_loader = DataLoader(full_dataset, batch_size=64, shuffle=True)

In [ ]:
# ==========================================
# 2. Deep Botanist CNN Architecture
# ==========================================
class BotanistCNN(nn.Module):
    def __init__(self):
        super(BotanistCNN, self).__init__()

        # Block 1: Capture edges and basic colors
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Block 2: Capture complex disease spots and textures
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Adaptive pooling to ensure 4x4 output
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))
        self.dropout = nn.Dropout(0.5)
        self.flatten = nn.Flatten()

        # Fully connected layers
        self.fc1 = nn.Linear(4096, 512)
        self.fc2 = nn.Linear(512, 38)

    def forward(self, x):
        x = torch.relu(self.bn1(self.conv1(x)))
        x = self.pool1(torch.relu(self.bn2(self.conv2(x))))
        x = torch.relu(self.bn3(self.conv3(x)))
        x = self.pool2(torch.relu(self.bn4(self.conv4(x))))
        x = self.adaptive_pool(x)
        x = self.flatten(x)
        x = self.dropout(x)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return torch.log_softmax(x, dim=1)

In [ ]:
# ==========================================
# 3. Model Training Setup
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device in use: {device}")
model = BotanistCNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.NLLLoss()

def train(model, train_loader, optimizer, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % 100 == 0:
            print(f"Train Epoch {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)}] Loss: {loss.item():.6f}")

In [ ]:
# ==========================================
# 4. Final Training Loop
# ==========================================
total_epochs = 19
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_epochs)

print("Starting training on full dataset...")
for epoch in range(1, total_epochs + 1):
    train(model, train_loader, optimizer, epoch)
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    print(f"Epoch {epoch} ended. Learning rate: {current_lr:.6f}\n")

# Save the trained weights
torch.save(model.state_dict(), CHECKPOINT_PATH)
print(f"Model saved to: {CHECKPOINT_PATH}")
print("Training complete!")